# NeurIPS 2026 Paper PDF Compilation — Evaluation Demo

**What this artifact does:** Compiles a complete NeurIPS 2026 paper PDF from prior experiment artifacts. It copies 8 pre-generated figures, generates 1 ablation bar chart, assembles LaTeX with 9 figures/5 tables/40 bib entries integrating ablation decomposition and routing simulation data, compiles via pdflatex+bibtex, and outputs 19 compilation quality metrics.

**This demo** loads the pre-computed evaluation results, reproduces the ablation figure generation, performs figure quality and dimension compliance analysis, and visualizes all compilation metrics.

In [1]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# No non-Colab packages needed — all imports are from stdlib or pre-installed packages

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'matplotlib==3.10.0')


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import json
import os
import re

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

In [3]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-276cb0-flickering-before-failing-ecological-cri/main/evaluation_iter7_neurips_2026_pa/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [4]:
data = load_data()
print(f"Loaded evaluation data: {data['metadata']['evaluation_name']}")
print(f"  Description: {data['metadata']['description']}")
print(f"  Datasets: {[d['dataset'] for d in data['datasets']]}")
print(f"  Total runtime: {data['metadata']['total_runtime_seconds']:.2f}s")

Loaded evaluation data: NeurIPS_Paper_PDF_Compilation_v2
  Description: Complete NeurIPS 2026 paper compilation with ablation decomposition and routing simulation integration. 9 figures, 5 tables, 38+ bib entries.
  Datasets: ['figure_quality', 'compilation_summary']
  Total runtime: 11.97s


In [5]:
# ── Configuration ──
# NeurIPS column widths (inches)
SINGLE_COL_WIDTH = 3.25
DOUBLE_COL_WIDTH = 6.75

# Targets for the paper compilation
N_FIGURES_TARGET = 9
N_TABLES_TARGET = 5
N_BIB_ENTRIES_TARGET = 38

# Ablation figure data (from eval_id2_it6 results)
ABLATION_CATEGORIES = [
    "Random\nBaseline",
    "Pure CSD\n(3 features)",
    "CSD +\nDynamics",
    "Difficulty\nOnly",
    "Full Model\n(CSD+Diff)",
]
ABLATION_F1_SCORES = [0.509, 0.690, 0.734, 0.886, 0.949]
ABLATION_COLORS = ["#999999", "#66b3ff", "#4d94ff", "#ff9966", "#2d6df6"]

# Figure metadata from the paper
FIGURE_INFO = [
    {"name": "fig1_csd_framework", "caption": "CSD Framework Overview",
     "data_points": 0, "width": "single", "w_in": 3.1, "h_in": 2.17},
    {"name": "fig2_accuracy_curves", "caption": "Accuracy vs Difficulty",
     "data_points": 132, "width": "single", "w_in": 3.05, "h_in": 3.7},
    {"name": "fig3_csd_indicators", "caption": "CSD Indicator Profiles",
     "data_points": 528, "width": "double", "w_in": 6.55, "h_in": 3.67},
    {"name": "fig4_temperature", "caption": "Temperature-Dependent CSD",
     "data_points": 16, "width": "double", "w_in": 6.55, "h_in": 3.02},
    {"name": "fig5_classifier_comparison", "caption": "Classifier Comparison",
     "data_points": 23, "width": "double", "w_in": 6.67, "h_in": 2.63},
    {"name": "fig6_model_fits_gc_gemini", "caption": "Model Fits",
     "data_points": 240, "width": "single", "w_in": 3.05, "h_in": 3.37},
    {"name": "fig7_aggregate_models", "caption": "Aggregate Model Comparison",
     "data_points": 22, "width": "double", "w_in": 6.55, "h_in": 2.5},
    {"name": "fig8_prospective_protocol", "caption": "Prospective Protocol",
     "data_points": 17, "width": "double", "w_in": 6.55, "h_in": 2.5},
]

## Section 1: Figure Quality Analysis

Extract per-figure quality metrics from the evaluation data and check dimension compliance against NeurIPS formatting requirements.

In [6]:
# Extract figure quality dataset from loaded data
figure_dataset = [d for d in data["datasets"] if d["dataset"] == "figure_quality"][0]
figure_examples = figure_dataset["examples"]

# Analyze each figure's quality metrics
figure_results = []
for fig in figure_examples:
    name = fig["metadata_figure_name"]
    size_kb = fig["eval_filesize_png_kb"]
    w_in = fig["eval_width_inches"]
    h_in = fig["eval_height_inches"]

    # Size check: 5 KB <= size <= 2048 KB
    size_ok = 5 <= size_kb <= 2048

    # Dimension compliance: width should be close to single-col or double-col
    dim_ok = (abs(w_in - SINGLE_COL_WIDTH) < 0.5) or (abs(w_in - DOUBLE_COL_WIDTH) < 0.5)

    figure_results.append({
        "name": name,
        "generated": bool(fig["eval_figure_generated"]),
        "size_kb": size_kb,
        "w_in": w_in,
        "h_in": h_in,
        "data_points": fig["eval_n_data_points"],
        "size_check_pass": size_ok,
        "dimension_compliance": dim_ok,
        "width_type": "single" if abs(w_in - SINGLE_COL_WIDTH) < abs(w_in - DOUBLE_COL_WIDTH) else "double",
    })
    status = "PASS" if (size_ok and dim_ok) else "FAIL"
    print(f"  {name}: {size_kb:.1f} KB, {w_in:.2f}x{h_in:.2f} in, {fig['eval_n_data_points']} pts [{status}]")

n_generated = sum(1 for f in figure_results if f["generated"])
n_pass_size = sum(1 for f in figure_results if f["size_check_pass"])
n_pass_dim = sum(1 for f in figure_results if f["dimension_compliance"])
print(f"\nSummary: {n_generated}/{len(figure_results)} generated, "
      f"{n_pass_size} pass size check, {n_pass_dim} pass dimension compliance")

  fig1_csd_framework: 111.2 KB, 3.10x2.17 in, 0 pts [PASS]
  fig2_accuracy_curves: 184.9 KB, 3.05x3.70 in, 132 pts [PASS]
  fig3_csd_indicators: 468.3 KB, 6.55x3.67 in, 528 pts [PASS]
  fig4_temperature: 108.4 KB, 6.55x3.02 in, 16 pts [PASS]
  fig5_classifier_comparison: 116.5 KB, 6.67x2.63 in, 23 pts [PASS]
  fig6_model_fits_gc_gemini: 136.1 KB, 3.05x3.37 in, 240 pts [PASS]
  fig7_aggregate_models: 110.3 KB, 6.55x2.50 in, 22 pts [PASS]
  fig8_prospective_protocol: 87.3 KB, 6.55x2.50 in, 17 pts [PASS]
  fig9_ablation_decomposition: 87.7 KB, 6.65x2.70 in, 5 pts [PASS]

Summary: 9/9 generated, 9 pass size check, 9 pass dimension compliance


## Section 2: Generate Ablation Bar Chart (fig9)

Reproduce the ablation decomposition figure from the original script. This chart shows LOPO F1 scores across five feature set configurations, demonstrating the unique contribution of CSD (Critical Slowing Down) indicators.

In [7]:
# Generate fig9: ablation bar chart (reproduced from eval.py)
fig, ax = plt.subplots(figsize=(DOUBLE_COL_WIDTH, 2.8), dpi=150)
bars = ax.bar(range(len(ABLATION_CATEGORIES)), ABLATION_F1_SCORES, color=ABLATION_COLORS,
              edgecolor="black", linewidth=0.5, width=0.7)

# Add value labels
for bar, val in zip(bars, ABLATION_F1_SCORES):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.015,
            f"{val:.3f}", ha="center", va="bottom", fontsize=8, fontweight="bold")

ax.set_xticks(range(len(ABLATION_CATEGORIES)))
ax.set_xticklabels(ABLATION_CATEGORIES, fontsize=7)
ax.set_ylabel("LOPO F1 Score", fontsize=9)
ax.set_ylim(0, 1.08)
ax.set_title("Feature Ablation: Isolating the Ecological Contribution", fontsize=10, fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.axhline(y=0.509, color="#999999", linestyle="--", alpha=0.5, linewidth=0.8)

# Annotation arrows showing contributions
ax.annotate("", xy=(4, 0.949), xytext=(3, 0.886),
            arrowprops=dict(arrowstyle="->", color="green", lw=1.5))
ax.text(3.7, 0.92, "+6.3pp\nCSD lift", fontsize=6, color="green", ha="center")

plt.tight_layout()
plt.show()
print("Ablation figure generated successfully.")

Ablation figure generated successfully.


## Section 3: Compilation Metrics Analysis

Extract and analyze the compilation summary metrics from the evaluation data, checking all 19 quality indicators.

In [8]:
# Extract compilation summary from loaded data
comp_dataset = [d for d in data["datasets"] if d["dataset"] == "compilation_summary"][0]
comp_example = comp_dataset["examples"][0]

# Extract aggregate metrics
metrics = data["metrics_agg"]

# Print compilation summary
print("=" * 60)
print("COMPILATION SUMMARY")
print("=" * 60)
print(f"  PDF compiled:              {bool(metrics['pdf_compiled'])}")
print(f"  Page count:                {metrics['page_count']}")
print(f"  Main body pages:           {metrics['main_body_pages']}")
print(f"  Figures included:          {metrics['total_figures_included']}/{N_FIGURES_TARGET}")
print(f"  Tables included:           {metrics['total_tables_included']}/{N_TABLES_TARGET}")
print(f"  Bibliography entries:      {metrics['bibliography_entries_total']}")
print(f"  Bibliography resolved:     {metrics['bibliography_entries_resolved']}")
print(f"  Missing citations:         {metrics['missing_citations']}")
print(f"  Undefined references:      {metrics['undefined_references']}")
print(f"  LaTeX warnings:            {metrics['latex_warnings_count']}")
print(f"  Overfull hbox:             {metrics['overfull_hbox_count']}")
print(f"  No placeholder text:       {bool(metrics['no_placeholder_text'])}")
print(f"  Compilation success rate:  {metrics['compilation_success_rate']:.1%}")
print(f"  All figures pass size:     {bool(metrics['all_figures_pass_size_check'])}")
print(f"  NeurIPS dim compliance:    {bool(metrics['neurips_dimension_compliance'])}")
print(f"  Total data points:         {metrics['total_data_points_in_figures']}")
print(f"  Ablation section present:  {bool(metrics['ablation_section_present'])}")
print(f"  Routing section present:   {bool(metrics['routing_section_present'])}")
print(f"  New tables present:        {metrics['new_tables_present']}")
print("=" * 60)

COMPILATION SUMMARY
  PDF compiled:              True
  Page count:                14
  Main body pages:           10
  Figures included:          9/9
  Tables included:           5/5
  Bibliography entries:      40
  Bibliography resolved:     36
  Missing citations:         0
  Undefined references:      0
  LaTeX warnings:            1
  Overfull hbox:             0
  No placeholder text:       True
  Compilation success rate:  100.0%
  All figures pass size:     True
  NeurIPS dim compliance:    True
  Total data points:         983
  Ablation section present:  True
  Routing section present:   True
  New tables present:        2


## Section 4: Dimension Compliance Check

Verify that all 9 figures meet NeurIPS formatting requirements: single-column width (~3.25 in) or double-column width (~6.75 in), with file sizes between 5 KB and 2048 KB.

In [9]:
# Dimension compliance analysis per figure
all_pass_size = True
all_comply_dims = True
total_data_points = 0

for fig in figure_results:
    if fig["generated"]:
        if not fig["size_check_pass"]:
            all_pass_size = False
        if not fig["dimension_compliance"]:
            all_comply_dims = False
        total_data_points += fig["data_points"]
    else:
        all_pass_size = False
        all_comply_dims = False

print(f"All figures pass size check:     {all_pass_size}")
print(f"NeurIPS dimension compliance:    {all_comply_dims}")
print(f"Total data points in figures:    {total_data_points}")
print()

# Classify figures by column width
single_col = [f for f in figure_results if f["width_type"] == "single"]
double_col = [f for f in figure_results if f["width_type"] == "double"]
print(f"Single-column figures ({SINGLE_COL_WIDTH} in target): {len(single_col)}")
for f in single_col:
    print(f"  {f['name']}: {f['w_in']:.2f} in (delta={abs(f['w_in'] - SINGLE_COL_WIDTH):.2f})")
print(f"Double-column figures ({DOUBLE_COL_WIDTH} in target): {len(double_col)}")
for f in double_col:
    print(f"  {f['name']}: {f['w_in']:.2f} in (delta={abs(f['w_in'] - DOUBLE_COL_WIDTH):.2f})")

All figures pass size check:     True
NeurIPS dimension compliance:    True
Total data points in figures:    983

Single-column figures (3.25 in target): 3
  fig1_csd_framework: 3.10 in (delta=0.15)
  fig2_accuracy_curves: 3.05 in (delta=0.20)
  fig6_model_fits_gc_gemini: 3.05 in (delta=0.20)
Double-column figures (6.75 in target): 6
  fig3_csd_indicators: 6.55 in (delta=0.20)
  fig4_temperature: 6.55 in (delta=0.20)
  fig5_classifier_comparison: 6.67 in (delta=0.08)
  fig7_aggregate_models: 6.55 in (delta=0.20)
  fig8_prospective_protocol: 6.55 in (delta=0.20)
  fig9_ablation_decomposition: 6.65 in (delta=0.10)


## Results Visualization

Comprehensive dashboard showing figure file sizes, dimension compliance, compilation quality scorecard, and data point distribution across all 9 figures.

In [10]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# ── Plot 1: Figure file sizes ──
ax1 = axes[0, 0]
names = [f["name"].replace("fig", "F").split("_")[0] for f in figure_results]
sizes = [f["size_kb"] for f in figure_results]
colors1 = ["#2d6df6" if f["size_check_pass"] else "#ff4444" for f in figure_results]
ax1.barh(names, sizes, color=colors1, edgecolor="black", linewidth=0.5)
ax1.axvline(x=5, color="red", linestyle="--", alpha=0.5, label="Min (5 KB)")
ax1.axvline(x=2048, color="red", linestyle="--", alpha=0.3, label="Max (2048 KB)")
ax1.set_xlabel("File Size (KB)")
ax1.set_title("Figure File Sizes", fontweight="bold")
ax1.legend(fontsize=7)
ax1.invert_yaxis()

# ── Plot 2: Figure dimensions ──
ax2 = axes[0, 1]
widths = [f["w_in"] for f in figure_results]
heights = [f["h_in"] for f in figure_results]
colors2 = ["#2d6df6" if f["dimension_compliance"] else "#ff4444" for f in figure_results]
ax2.scatter(widths, heights, c=colors2, s=80, edgecolor="black", linewidth=0.5, zorder=3)
for i, f in enumerate(figure_results):
    ax2.annotate(names[i], (widths[i], heights[i]), fontsize=6,
                 textcoords="offset points", xytext=(5, 5))
ax2.axvline(x=SINGLE_COL_WIDTH, color="green", linestyle="--", alpha=0.5, label=f"Single col ({SINGLE_COL_WIDTH}\")")
ax2.axvline(x=DOUBLE_COL_WIDTH, color="orange", linestyle="--", alpha=0.5, label=f"Double col ({DOUBLE_COL_WIDTH}\")")
ax2.set_xlabel("Width (inches)")
ax2.set_ylabel("Height (inches)")
ax2.set_title("Figure Dimensions vs NeurIPS Targets", fontweight="bold")
ax2.legend(fontsize=7)

# ── Plot 3: Quality scorecard ──
ax3 = axes[1, 0]
quality_labels = [
    "PDF Compiled", "Figures (9/9)", "Tables (5/5)",
    "No Missing Citations", "No Placeholders",
    "Size Check Pass", "Dim Compliance",
    "Ablation Section", "Routing Section",
]
quality_values = [
    metrics["pdf_compiled"],
    1 if metrics["total_figures_included"] == N_FIGURES_TARGET else 0,
    1 if metrics["total_tables_included"] == N_TABLES_TARGET else 0,
    1 if metrics["missing_citations"] == 0 else 0,
    metrics["no_placeholder_text"],
    metrics["all_figures_pass_size_check"],
    metrics["neurips_dimension_compliance"],
    metrics["ablation_section_present"],
    metrics["routing_section_present"],
]
colors3 = ["#27ae60" if v else "#e74c3c" for v in quality_values]
ax3.barh(quality_labels, quality_values, color=colors3, edgecolor="black", linewidth=0.5)
ax3.set_xlim(-0.1, 1.3)
ax3.set_title("Compilation Quality Scorecard", fontweight="bold")
for i, v in enumerate(quality_values):
    ax3.text(v + 0.05, i, "PASS" if v else "FAIL", va="center", fontsize=8, fontweight="bold")
ax3.invert_yaxis()

# ── Plot 4: Data points per figure ──
ax4 = axes[1, 1]
data_points = [f["data_points"] for f in figure_results]
colors4 = ["#2d6df6" if dp > 0 else "#cccccc" for dp in data_points]
ax4.bar(names, data_points, color=colors4, edgecolor="black", linewidth=0.5)
ax4.set_ylabel("Data Points")
ax4.set_title(f"Data Points per Figure (Total: {sum(data_points)})", fontweight="bold")
ax4.tick_params(axis="x", rotation=45)

plt.suptitle(f"NeurIPS 2026 Paper Compilation — {metrics['page_count']} pages, "
             f"{metrics['total_figures_included']} figures, {metrics['total_tables_included']} tables",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# ── Final summary ──
print("\n" + "=" * 60)
print("FINAL EVALUATION RESULTS")
print("=" * 60)
passed = sum(quality_values)
total = len(quality_values)
print(f"  Quality checks passed: {passed}/{total}")
print(f"  Page count: {metrics['page_count']} ({metrics['main_body_pages']} main body)")
print(f"  Bibliography: {metrics['bibliography_entries_resolved']}/{metrics['bibliography_entries_total']} resolved")
print(f"  Total data points in figures: {metrics['total_data_points_in_figures']}")
print(f"  Runtime: {data['metadata']['total_runtime_seconds']:.2f}s")
print("=" * 60)


FINAL EVALUATION RESULTS
  Quality checks passed: 9/9
  Page count: 14 (10 main body)
  Bibliography: 36/40 resolved
  Total data points in figures: 983
  Runtime: 11.97s
